# 16. OBB Detection - yolo26n-obb

| 항목 | 내용 |
|---|---|
| 대상 모델 | `yolo26n-obb.dxnn` |
| 카테고리 | **OBB Detection** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 겹침 정리 단계가 없는 이유를 출력 구조와 연결해 설명할 수 있다.
2. 회전 사각형의 다섯 값에서 네 꼭짓점을 계산할 수 있다.
3. 각도를 좁은 구간으로 맞추면서 가로·세로를 맞바꾸는 규칙을 설명할 수 있다.
4. 중심 좌표와 크기의 복원 계산이 다른 이유를 여백과 연결해 설명할 수 있다.

## 1. OBB Detection (회전 사각형 검출)

**영상에서 대상을 찾아 기울어진 사각형으로 위치·크기·방향을 함께 답하는 작업**

축에 나란한 사각형은 기울어진 대상을 감싸면서 빈 자리를 많이 포함한다. 나란히 세워 둔 차량이나 배처럼 **길고
기울어진 대상이 붙어 있으면** 축 정렬 사각형끼리 크게 겹쳐 서로를 지운다. 회전 사각형은 그 문제를 각도 하나를
더 답하는 것으로 푼다.

**출력의 성격**

사각형마다 중심·크기·각도가 붙는 목록이다. 각도가 있으므로 같은 대상을 여러 방식으로 적을 수 있어, 표기를 한
가지로 맞추는 정규화가 필요하다.

**활용 사례**

- 항공·위성 영상의 차량·선박·건물 판독
- 컨베이어 위 기울어진 부품 위치 파악
- 문서 속 기울어진 글자 영역 검출

**포함되지 않는 작업**

- 윤곽을 픽셀 단위로 나누기 (Instance Segmentation)
- 축에 나란한 사각형만 답하기 (Object Detection)
- 3차원 자세 추정 (별도 작업)

## 2. 전체 파이프라인

```text
dota8_test/P0284.png  1164×1196 BGR (항공 사진)
  → 전처리         BGR→RGB · 비율 유지 축소 997×1024 · 좌우 여백 13 → 1024×1024
  → run()          입력 Tensor 하나를 전달
  → npu_0          one2one 헤드의 중간 tensor 9개 (내부 전달, Public 아님)
  → cpu_0          output0  [1, 300, 7]  float32  (Public Output — 이미 정리된 300개)
  → 후처리         점수 임계값 → 좌표·크기 복원 → 각도 정규화 → 부류 이름 붙이기
  → Result         OBBResult (cx · cy · width · height · angle · confidence · class)
  → 시각화         다섯 값에서 네 꼭짓점을 계산해 기울어진 사각형을 그림
```

![16. OBB Detection 전체 파이프라인](resources/16_OBB_Detection_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `images` |
| Shape / Dtype | `[1, 1024, 1024, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **Letterbox — 비율 유지 후 여백 채우기** |
| Normalization | 없음 |

**입력이 1024×1024 로 검출 계열 중 가장 크다.** 항공 사진은 대상이 작고 많아 640 으로 줄이면 차량 한 대가 몇
화소로 뭉개진다. 이번 실행의 `gain` 은 `0.856187` 이고 여백은 **좌우에 13**씩이다 — 원본이 세로로 조금 더 길어
가로가 남았다.

### 3.2 출력

이 모델은 Task 가 둘이며 **후보 정리까지 모델 안에서 끝난다.**

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `one2one_cv2·cv3·cv4` 9개 | `[1, 4·15·1, 격자]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `output0` | `[1, 300, 7]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

`parse_model` 은 `cpu_0` 의 출력으로 `output0` 과 함께 `TopK_output_0` 도 보고하는데, **런타임이 목록에 내놓는
것은 `output0` 하나뿐**이다. 그래서 후처리는 `outputs[0]` 만 읽는다.

**300개는 고정이고 겹침 정리가 필요 없다.** 앞의 검출 모델들은 후보 수천~수만 개를 내놓고 NMS 로 겹친 것을
지웠는데, 이 모델은 **중복을 스스로 없애도록 학습된 헤드**(`one2one`)를 써서 정리된 목록 300개를 바로 낸다.
후처리에 NMS 단계가 없는 이유가 이것이며, 점수가 낮은 것을 버리는 일만 남는다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 300 | 결과 자리 수. 실제 대상 수와 무관하게 고정 |
| 2 | 7 | 결과 하나의 항목 수 |

**결과 한 줄 7개 값의 자리**

| index | 내용 | 이번 실행 범위 |
|---:|---|---|
| 0 | 중심 `cx` (입력 1024 기준) | — |
| 1 | 중심 `cy` | — |
| 2 | 폭 `w` (회전 전) | — |
| 3 | 높이 `h` (회전 전) | — |
| 4 | 점수 | `0.000166` ~ `0.942507` |
| 5 | **부류 번호** (실수로 담겨 있음) | `0.0` ~ `10.0` |
| 6 | 각도 (라디안) | `-0.6127` ~ `0.7704` |

부류 번호가 정수가 아니라 `float32` 자리에 담겨 있어 읽을 때 정수로 바꿔야 한다. 300개 자리 중 대부분은 점수가
0 에 가까운 빈 자리이며, 이번 실행에서 `0.3` 을 넘는 것은 **46개**다.

**각도는 라디안이고 좁은 구간으로 맞춘다.** 같은 사각형을 90도 돌리고 가로·세로를 맞바꾸면 완전히 같은 도형이
되므로 표기가 여러 개 생긴다. 그래서 각도를 `[0, π/2)` 로 맞추고, 그 과정에서 `π/2` 를 넘는 각도는 **가로와
세로를 맞바꿔** 같은 도형을 유지한다.

```text
t ← t mod π                     먼저 0~π 로
t ≥ π/2 이면  w ↔ h 를 맞바꿈
t ← t mod (π/2)                 0~π/2 로
```

점수를 통과한 46개는 각도가 모두 `[0, π/2)` 안에 들어와 **맞바꾼 개체가 0개**다. 반대로 `-0.6127` 같은 음수
각도는 `mod π` 로 `π - 0.6127 = 2.53` 이 되어 `π/2` 를 넘으므로, **이때는 가로와 세로를 맞바꾸고** 각도가
`0.96` 이 된다.

**네 꼭짓점은 두 벡터로 만든다.** 중심에서 가로 방향으로 `w/2`, 세로 방향으로 `h/2` 만큼 가는 벡터를 각도로
돌린 뒤 더하고 빼면 네 꼭짓점이 나온다.

```text
v1 = ( w/2·cos t ,  w/2·sin t )      가로 방향
v2 = (-h/2·sin t ,  h/2·cos t )      세로 방향 (v1 과 직각)

꼭짓점 = 중심 ± v1 ± v2   (네 조합)
```

**15개 부류의 번호**

번호의 정본은 후처리 코드가 들고 있는 이름 배열이다. DOTA 공식 페이지는 부류 이름만 나열하고 번호를 붙이지
않으므로, 이 배열의 순서가 곧 번호다.

| index | 이름 | index | 이름 | index | 이름 |
|---:|---|---:|---|---:|---|
| 0 | plane | 5 | basketball-court | 10 | small-vehicle |
| 1 | ship | 6 | ground-track-field | 11 | helicopter |
| 2 | storage-tank | 7 | harbor | 12 | roundabout |
| 3 | baseball-diamond | 8 | bridge | 13 | soccer-ball-field |
| 4 | tennis-court | 9 | large-vehicle | 14 | swimming-pool |

세로로 읽는 표가 아니다. 0~14 를 다섯 개씩 세 묶음으로 옆에 늘어놓았다.

전체 목록: [ultralytics DOTAv1.yaml](https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/DOTAv1.yaml)

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `cpu_0` 이 `[1, 300, 7]`
을 내놓고 중간 tensor 이름에 `one2one` 이 붙어 있는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "yolo26n-obb.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `score_threshold` | 0.3 | 0.3 | 이 점수에 못 미치는 결과를 버림 |

겹침 임계값이 없다. 모델이 중복을 스스로 정리하므로 NMS 를 쓰지 않고, 따라서 조절할 값도 없다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 설정 파일과 DOTA 15개 이름을 정의한다. 이후 셀은 여기서 만든
이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "dota8_test/P0284.png"
CONFIG_PATH = DX_APP / "src" / "python_example" / "obb_detection" / "yolo26n_obb" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

DOTA_LABELS = [
    "plane", "ship", "storage-tank", "baseball-diamond", "tennis-court",
    "basketball-court", "ground-track-field", "harbor", "bridge",
    "large-vehicle", "small-vehicle", "helicopter", "roundabout",
    "soccer-ball-field", "swimming-pool",
]

print("config", CONFIG)
print("classes", len(DOTA_LABELS), "/ index 9 =", DOTA_LABELS[9])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. `scale`(gain)·`pad_x`·`pad_y` 가
  좌표 복원의 근거다
- `OBBResult` — 후처리가 만들어 시각화로 넘기는 결과. 중심·크기·각도·점수·부류를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale: float = 1.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class OBBResult:
    """회전 사각형 하나의 중심·크기·각도·점수·부류. 원본 대응: common/base/i_processor.py"""

    cx: float = 0.0
    cy: float = 0.0
    width: float = 0.0
    height: float = 0.0
    angle: float = 0.0
    confidence: float = 0.0
    class_id: int = 0
    class_name: str = ""


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. `parse_model` 이 둘을 보고하는데도 목록에
`output0` 하나만 나오는 것을 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output (런타임 순서):")
for index, info in enumerate(output_info):
    print(f"  [{index}] {info['name']:12s} {info['shape']}")

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

비율을 유지해 997×1024 로 줄이고 좌우에 13씩 회색 여백을 넣어 1024×1024 를 만든다. 원본이 세로로 조금 더 길어
**이번에는 가로에 여백이 생긴다** — 앞의 검출 모델들과 여백이 붙는 방향이 반대다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 1024×1024, 그중 실제 사진은 997×1024
- 여백 — 좌우 13씩, 위아래 0
- Context — `scale`(gain) 과 `pad_x`·`pad_y`

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 항공 사진이라 대상이 작고 방향이 제각각이다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(8, 8))

#### 전처리 함수 정의

- `obb_preprocessor_create` — 모델 입력 크기와 여백 색을 state 에 담음
- `obb_letterbox` — 비율 유지 축소와 여백 채우기
- `obb_preprocessor_process` — letterbox 를 적용하고 복원용 값을 Context 에 담음

In [ ]:
def obb_preprocessor_create(input_width: int, input_height: int,
                            pad_color: tuple = (114, 114, 114)) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height, "pad_color": pad_color}


def obb_letterbox(state: dict, image: np.ndarray) -> tuple:
    """비율을 유지해 줄이고 남는 자리를 채운다. 원본 대응: LetterboxPreprocessor._letterbox()"""
    height, width = image.shape[:2]
    gain = min(state["input_height"] / height, state["input_width"] / width)
    new_size = int(round(width * gain)), int(round(height * gain))

    delta_w = (state["input_width"] - new_size[0]) / 2
    delta_h = (state["input_height"] - new_size[1]) / 2
    if (width, height) != new_size:
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(delta_h - 0.1)), int(round(delta_h + 0.1))
    left, right = int(round(delta_w - 0.1)), int(round(delta_w + 0.1))
    padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT,
                                value=state["pad_color"])
    return padded, gain, (top, left)


def obb_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    padded, gain, pad = obb_letterbox(state, rgb)
    ctx.scale = gain
    ctx.pad_y, ctx.pad_x = pad
    return padded, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. `pad_x` 가 0 이 아니고 `pad_y` 가 0 인 것을 확인한다.

In [ ]:
preprocessor_state = obb_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = obb_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"gain={ctx.scale:.6f} pad_x={ctx.pad_x} pad_y={ctx.pad_y}")

notebook_show_bgr(cv2.cvtColor(input_tensor, cv2.COLOR_RGB2BGR),
                  f"Letterboxed {INPUT_WIDTH}x{INPUT_HEIGHT} (gray bands on left and right)",
                  figsize=(7, 7))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이 포함돼
있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 결과 자리 수 — 300개 고정
- 항목별 범위 — 점수·부류 번호·각도가 각각 어느 구간인지
- 점수 통과 개수 — 300개 중 몇 개가 실제 대상인지
- 부류 번호의 형식 — 정수가 아니라 실수 자리에 담겨 있다

In [ ]:
raw = np.squeeze(np.asarray(outputs[0]))
notebook_describe_array("output0", raw)

score_threshold = CONFIG.get("score_threshold", 0.3)
print(f"결과 자리 {raw.shape[0]}개 × 항목 {raw.shape[1]}개 = cx cy w h score class angle")
print(f"  점수      {raw[:, 4].min():.6f} ~ {raw[:, 4].max():.6f}")
print(f"  부류 번호  {raw[:, 5].min():.1f} ~ {raw[:, 5].max():.1f}  dtype={raw.dtype} (실수 자리)")
print(f"  각도      {raw[:, 6].min():.4f} ~ {raw[:, 6].max():.4f} rad "
      f"({np.degrees(raw[:, 6].min()):.1f}° ~ {np.degrees(raw[:, 6].max()):.1f}°)")
print(f"\n{score_threshold} 초과 {int((raw[:, 4] >= score_threshold).sum())}개 / {raw.shape[0]}개")
print("첫 줄 원본 값:", np.array2string(raw[0], precision=4))

### 4.4 후처리

점수로 걸러 내고, 중심과 크기를 원본 좌표계로 되돌리고, 각도를 좁은 구간으로 맞춘 뒤 부류 이름을 붙인다.
**겹침 정리 단계가 없다.**

**중심과 크기의 복원 계산이 다르다.** 중심은 여백을 뺀 뒤 `gain` 으로 나누지만, 크기는 **`gain` 으로만** 나눈다.
여백은 그림을 옮겨 놓기만 했고 늘리거나 줄이지 않았으므로 크기에는 영향이 없다.

```text
원본 cx = (입력 cx - pad_x) / gain      원본 w = 입력 w / gain
원본 cy = (입력 cy - pad_y) / gain      원본 h = 입력 h / gain
```

**각도 정규화는 표기를 하나로 맞추는 일이다.** 같은 도형을 여러 각도로 적을 수 있으므로 `[0, π/2)` 로 맞추고,
그 과정에서 가로·세로를 맞바꿔야 도형이 그대로 유지된다. 맞바꾸는 것을 잊으면 사각형이 90도 돌아간다.

#### 후처리 함수 정의

- `obb_postprocessor_create` — 임계값과 이름 배열을 state 에 담음
- `obb_regularize_boxes` — 각도를 `[0, π/2)` 로 맞추고 필요하면 가로·세로를 맞바꿈
- `obb_postprocessor_process` — 걸러내기·복원·정규화를 거쳐 Result 생성

In [ ]:
def obb_postprocessor_create(config: dict, labels: list) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"score_threshold": config.get("score_threshold", 0.3),
            "labels": config.get("labels", labels)}


def obb_regularize_boxes(boxes: np.ndarray) -> np.ndarray:
    """각도를 [0, π/2) 로 맞추고 필요하면 가로·세로를 맞바꾼다.

    원본 대응: OBBPostprocessor._regularize_rboxes()
    """
    if boxes.size == 0:
        return boxes
    cx, cy, width, height, angle = np.split(boxes, 5, axis=-1)
    angle = np.mod(angle, np.pi)
    swap = angle >= (np.pi / 2)
    return np.concatenate([cx, cy,
                           np.where(swap, height, width),
                           np.where(swap, width, height),
                           np.mod(angle, np.pi / 2)], axis=-1)


def obb_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    output = np.squeeze(np.asarray(outputs[0]))
    if output.ndim != 2 or output.shape[1] < 7:
        return []

    scores = output[:, 4]
    kept = scores >= state["score_threshold"]
    if not np.any(kept):
        return []
    filtered = output[kept]

    gain = max(ctx.scale, 1e-6)
    # 중심은 여백을 뺀 뒤 나누고, 크기는 나누기만 한다.
    boxes = np.column_stack([
        (filtered[:, 0] - ctx.pad_x) / gain,
        (filtered[:, 1] - ctx.pad_y) / gain,
        filtered[:, 2] / gain,
        filtered[:, 3] / gain,
        filtered[:, 6],
    ])
    boxes = obb_regularize_boxes(boxes)

    if ctx.original_width > 0 and ctx.original_height > 0:
        boxes[:, 0] = np.clip(boxes[:, 0], 0, ctx.original_width - 1)
        boxes[:, 1] = np.clip(boxes[:, 1], 0, ctx.original_height - 1)

    results = []
    for index in range(len(boxes)):
        class_id = int(filtered[index, 5])
        name = (state["labels"][class_id] if 0 <= class_id < len(state["labels"])
                else f"class_{class_id}")
        results.append(OBBResult(cx=float(boxes[index, 0]), cy=float(boxes[index, 1]),
                                 width=float(boxes[index, 2]), height=float(boxes[index, 3]),
                                 angle=float(boxes[index, 4]),
                                 confidence=float(filtered[index, 4]),
                                 class_id=class_id, class_name=name))
    return results

#### 후처리 실행

확인할 것:

- 결과 개수 — 점수를 넘은 것만 남는지
- 부류 분포 — 항공 사진에서 어떤 부류가 나왔는지
- 각도 — 도(degree)로 바꿨을 때 몇 도인지
- 가로·세로를 맞바꾼 개체 — 이번 실행에서 몇 개인지

In [ ]:
postprocessor_state = obb_postprocessor_create(CONFIG, DOTA_LABELS)
results = obb_postprocessor_process(postprocessor_state, outputs, ctx)

print(f"결과 자리 {raw.shape[0]}개 → 점수 {postprocessor_state['score_threshold']} 통과 "
      f"{len(results)}개 (겹침 정리 없음)")

counts = {}
for result in results:
    counts[result.class_name] = counts.get(result.class_name, 0) + 1
print("부류별:", ", ".join(f"{name} {count}개" for name, count in
                          sorted(counts.items(), key=lambda pair: -pair[1])))

print("\n점수 상위 6개:")
for result in sorted(results, key=lambda r: -r.confidence)[:6]:
    print(f"  {result.class_name:14s} score={result.confidence:.4f} "
          f"중심=({result.cx:6.1f}, {result.cy:6.1f}) 크기={result.width:5.1f}x{result.height:5.1f} "
          f"각도={np.degrees(result.angle):5.1f}°")

swapped = int((np.mod(raw[raw[:, 4] >= postprocessor_state["score_threshold"], 6], np.pi)
               >= np.pi / 2).sum())
print(f"\n각도 정규화로 가로·세로를 맞바꾼 개체 {swapped}개 / {len(results)}개")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 다섯 값에서 네 꼭짓점을 계산하고 기울어진 사각형을 그린다. 라벨은 사각형의 가장 왼쪽 위 좌표에
붙인다. **원본은 부류마다 색을 무작위로 뽑으므로 실행할 때마다 색이 달라진다** — 노트북에서는 같은 그림이 남도록
난수 씨앗을 고정한다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가
깨진다.

In [ ]:
def obb_corners(cx: float, cy: float, width: float, height: float, angle: float) -> np.ndarray:
    """중심·크기·각도에서 네 꼭짓점을 계산한다. 원본 대응: OBBVisualizer._xywhr_to_corners()"""
    cos_value, sin_value = np.cos(angle), np.sin(angle)
    vector_w = np.array([width / 2 * cos_value, width / 2 * sin_value])
    vector_h = np.array([-height / 2 * sin_value, height / 2 * cos_value])
    center = np.array([cx, cy])
    return np.array([center + vector_w + vector_h, center + vector_w - vector_h,
                     center - vector_w - vector_h, center - vector_w + vector_h],
                    dtype=np.float32)


def obb_visualizer_create(labels: list, seed: int = 0) -> dict:
    """부류별 색을 만들어 state 에 담는다. 원본 대응: Factory.create_visualizer()"""
    generator = np.random.default_rng(seed)
    return {"labels": labels, "palette": generator.uniform(0, 255, size=(len(labels), 3))}


def obb_visualizer_visualize(state: dict, image: np.ndarray, results: list,
                             draw_label: bool = True) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()

    for result in results:
        polygon = obb_corners(result.cx, result.cy, result.width, result.height,
                              result.angle).astype(np.int32)
        color = state["palette"][result.class_id % len(state["palette"])]
        cv2.polylines(output, [polygon], True, color, 2)

        if not draw_label:
            continue
        label = f"{result.class_name}: {result.confidence:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        box_x, box_y = int(polygon[:, 0].min()), int(polygon[:, 1].min())
        label_y = box_y - 10 if box_y - 10 > label_h else box_y + 10
        cv2.rectangle(output, (box_x, label_y - label_h), (box_x + label_w, label_y + label_h),
                      color, cv2.FILLED)
        cv2.putText(output, label, (box_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1,
                    cv2.LINE_AA)

    return output

#### 결과 시각화

회전 사각형을 inline 으로 표시한다. 대상이 많아 라벨이 서로 겹치므로 사각형만 그린 그림도 함께 본다.

In [ ]:
visualizer_state = obb_visualizer_create(DOTA_LABELS)
output_image = obb_visualizer_visualize(visualizer_state, image, results)
notebook_show_bgr(output_image, f"{len(results)} oriented boxes", figsize=(9, 9))

notebook_show_bgr(obb_visualizer_visualize(visualizer_state, image, results, draw_label=False),
                  "Boxes only (no labels)", figsize=(9, 9))

#### 축에 나란한 사각형과 비교

회전 사각형을 쓰는 이유는 기울어진 대상을 감쌀 때 빈 자리가 적기 때문이다. 같은 결과를 축에 나란한 사각형으로
바꿔 그려 면적을 비교하면 그 차이가 수치로 드러난다 — 이 사진에서는 축에 나란히 감쌀 때 면적이 **2.18배**가
된다. 그만큼이 대상 아닌 배경이며, 옆에 붙어 있는 차량끼리 서로를 지우게 되는 원인이다.

In [ ]:
rotated_area = axis_area = 0.0
comparison = image.copy()
for result in results:
    polygon = obb_corners(result.cx, result.cy, result.width, result.height, result.angle)
    x1, y1 = polygon[:, 0].min(), polygon[:, 1].min()
    x2, y2 = polygon[:, 0].max(), polygon[:, 1].max()
    rotated_area += result.width * result.height
    axis_area += (x2 - x1) * (y2 - y1)
    cv2.rectangle(comparison, (int(x1), int(y1)), (int(x2), int(y2)), (60, 60, 220), 2)
    cv2.polylines(comparison, [polygon.astype(np.int32)], True, (0, 220, 60), 2)

print(f"회전 사각형 면적 합 {rotated_area:12,.0f}")
print(f"축 정렬 사각형 면적 합 {axis_area:12,.0f} ({axis_area / rotated_area:.2f}배)")
print("초록이 회전 사각형, 빨강이 같은 대상을 축에 나란히 감싼 사각형이다.")
notebook_show_bgr(comparison, "Rotated (green) vs axis-aligned (red)", figsize=(9, 9))